# Business Validation

Compares only:
1. best model
2. baseline model
3. do_nothing

Financial rule:
- if claim + claim type are both correctly predicted, realized claim cost is reduced by 50%
- otherwise full cost is paid


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path('..').resolve() / 'data'

# Preferred files (strategy-level, exported by ML notebook)
VAL_BUSINESS_PATH = BASE_DIR / 'val_business_predictions.parquet'
TEST_BUSINESS_PATH = BASE_DIR / 'test_business_predictions.parquet'

# Fallback files (main-model only)
VAL_CLAIM_PATH = BASE_DIR / 'val_claim_predictions.parquet'
TEST_CLAIM_PATH = BASE_DIR / 'test_claim_predictions.parquet'
VAL_TYPE_PATH = BASE_DIR / 'val_claim_type_predictions.parquet'
TEST_TYPE_PATH = BASE_DIR / 'test_claim_type_predictions.parquet'

if VAL_BUSINESS_PATH.exists() and TEST_BUSINESS_PATH.exists():
    val_pred = pd.read_parquet(VAL_BUSINESS_PATH).copy()
    test_pred = pd.read_parquet(TEST_BUSINESS_PATH).copy()
    source_mode = 'business_predictions'
elif all(p.exists() for p in [VAL_CLAIM_PATH, TEST_CLAIM_PATH, VAL_TYPE_PATH, TEST_TYPE_PATH]):
    # Build a compatible dataframe from main-model files only
    v_claim = pd.read_parquet(VAL_CLAIM_PATH).copy()
    t_claim = pd.read_parquet(TEST_CLAIM_PATH).copy()
    v_type = pd.read_parquet(VAL_TYPE_PATH).copy()
    t_type = pd.read_parquet(TEST_TYPE_PATH).copy()

    # merge type predictions on resident/week; non-claim predictions become NoClaim
    val_pred = v_claim.merge(
        v_type[['resident_id', 'week_start', 'y_type_true', 'y_type_pred']],
        on=['resident_id', 'week_start'],
        how='left'
    )
    test_pred = t_claim.merge(
        t_type[['resident_id', 'week_start', 'y_type_true', 'y_type_pred']],
        on=['resident_id', 'week_start'],
        how='left'
    )

    for d in [val_pred, test_pred]:
        if 'y_type_true' not in d.columns:
            d['y_type_true'] = 'NoClaim'
        if 'y_type_pred' not in d.columns:
            d['y_type_pred'] = 'NoClaim'
        d['y_type_true'] = d['y_type_true'].fillna('NoClaim')
        d['y_type_pred'] = d['y_type_pred'].fillna('NoClaim')
        d['strategy'] = 'main_model_only'

    source_mode = 'main_only_fallback'
else:
    raise FileNotFoundError(
        'Missing prediction files. Run prototype/weekly_claims_ml.ipynb export cell. '        f'Expected either {VAL_BUSINESS_PATH.name}/{TEST_BUSINESS_PATH.name} or claim+type fallback files.'
    )

val_pred['split'] = 'val'
test_pred['split'] = 'test'
pred = pd.concat([val_pred, test_pred], ignore_index=True)
pred['week_start'] = pd.to_datetime(pred['week_start'], errors='coerce')

# Safety fallback: if we only have one strategy, synthesize a no-action baseline
# so best-vs-baseline comparison always works in this notebook.
if pred['strategy'].dropna().nunique() < 2:
    baseline = pred.copy()
    baseline['strategy'] = 'no_action_baseline'
    baseline['y_claim_pred'] = 0
    baseline['y_type_pred'] = 'NoClaim'
    pred = pd.concat([pred, baseline], ignore_index=True)
    print('info: synthesized fallback strategy -> no_action_baseline')

print('source_mode:', source_mode)
print('rows:', len(pred))
print('strategies available:', sorted(pred['strategy'].dropna().unique().tolist()))
pred.head()


source_mode: main_only_fallback
rows: 19368
strategies available: ['main_model_only']


,resident_id,week_start,y_claim_true,p_claim_pred,y_claim_pred,threshold_used,y_type_true,y_type_pred,strategy,split
0,000d4288-4983-5512-b4e8-bf5efa8e20ef,2024-08-12,0,0.038423,0,0.075,NoClaim,NoClaim,main_model_only,val
1,000d4288-4983-5512-b4e8-bf5efa8e20ef,2024-08-19,0,0.016262,0,0.075,NoClaim,NoClaim,main_model_only,val
2,000d4288-4983-5512-b4e8-bf5efa8e20ef,2024-08-26,0,0.017579,0,0.075,NoClaim,NoClaim,main_model_only,val
3,000d4288-4983-5512-b4e8-bf5efa8e20ef,2024-09-02,0,0.014929,0,0.075,NoClaim,NoClaim,main_model_only,val
4,000d4288-4983-5512-b4e8-bf5efa8e20ef,2024-09-09,0,0.018369,0,0.075,NoClaim,NoClaim,main_model_only,val


In [2]:
# README cost assumptions
claim_type_cost_map = {
    'falls': 3500.0,
    'medication errors': 5000.0,
    'wounds / pressure injuries': 4000.0,
    'return-to-hospital (rth) events': 20000.0,
    'elopement / wandering': 2500.0,
    'altercations': 2500.0,
}

claim_type_aliases = {
    'fall': 'falls', 'falls': 'falls',
    'medication error': 'medication errors', 'medication errors': 'medication errors',
    'wound': 'wounds / pressure injuries', 'pressure injury': 'wounds / pressure injuries',
    'wounds': 'wounds / pressure injuries', 'wounds / pressure injuries': 'wounds / pressure injuries',
    'rth': 'return-to-hospital (rth) events', 'return to hospital': 'return-to-hospital (rth) events',
    'return-to-hospital': 'return-to-hospital (rth) events', 'return-to-hospital (rth) events': 'return-to-hospital (rth) events',
    'elopement': 'elopement / wandering', 'wandering': 'elopement / wandering', 'elopement / wandering': 'elopement / wandering',
    'altercation': 'altercations', 'altercations': 'altercations',
}

unknown_claim_default_cost = (
    13*3500 + 10*5000 + 7*4000 + 7*20000 + 5*2500 + 2*2500
) / (13 + 10 + 7 + 7 + 5 + 2)


def normalize_claim_type(v):
    s = str(v).strip().lower()
    if s in {'', 'nan', 'none', 'noclaim'}:
        return 'NoClaim'
    if s in claim_type_aliases:
        return claim_type_aliases[s]
    if 'fall' in s:
        return 'falls'
    if 'medication' in s:
        return 'medication errors'
    if 'pressure' in s or 'wound' in s:
        return 'wounds / pressure injuries'
    if 'rth' in s or 'return' in s or 'hospital' in s:
        return 'return-to-hospital (rth) events'
    if 'elop' in s or 'wander' in s:
        return 'elopement / wandering'
    if 'altercation' in s or 'fight' in s:
        return 'altercations'
    return 'Unknown'


def claim_cost_from_type(v):
    n = normalize_claim_type(v)
    if n == 'NoClaim':
        return 0.0
    if n in claim_type_cost_map:
        return float(claim_type_cost_map[n])
    return float(unknown_claim_default_cost)


In [3]:
# Normalize and compute financial impact for each strategy row
pred['y_claim_true'] = pred['y_claim_true'].astype(int)
pred['y_claim_pred'] = pred['y_claim_pred'].astype(int)

pred['y_type_true_norm'] = pred['y_type_true'].map(normalize_claim_type)
pred['y_type_pred_norm'] = pred['y_type_pred'].map(normalize_claim_type)

pred.loc[pred['y_claim_true'] == 0, 'y_type_true_norm'] = 'NoClaim'
pred.loc[pred['y_claim_pred'] == 0, 'y_type_pred_norm'] = 'NoClaim'

pred['actual_cost'] = pred['y_type_true_norm'].map(claim_cost_from_type).astype(float)
pred['mitigated'] = (
    (pred['y_claim_true'] == 1)
    & (pred['y_claim_pred'] == 1)
    & (pred['y_type_pred_norm'] == pred['y_type_true_norm'])
)
pred['strategy_cost'] = np.where(pred['mitigated'], 0.5 * pred['actual_cost'], pred['actual_cost'])


In [4]:
# Pick BEST and BASELINE based on validation financial result
val_cost_by_strategy = (
    pred[pred['split'] == 'val']
    .groupby('strategy', as_index=False)['strategy_cost']
    .sum()
    .sort_values('strategy_cost', ascending=True)
)

if len(val_cost_by_strategy) < 2:
    # Last-resort safety: add a no-action baseline and recompute ranking.
    b = pred.copy()
    b['strategy'] = 'no_action_baseline'
    b['y_claim_pred'] = 0
    b['y_type_pred'] = 'NoClaim'
    pred = pd.concat([pred, b], ignore_index=True)
    pred['y_type_pred_norm'] = pred['y_type_pred'].map(normalize_claim_type)
    pred.loc[pred['y_claim_pred'] == 0, 'y_type_pred_norm'] = 'NoClaim'
    pred['mitigated'] = (
        (pred['y_claim_true'] == 1)
        & (pred['y_claim_pred'] == 1)
        & (pred['y_type_pred_norm'] == pred['y_type_true_norm'])
    )
    pred['strategy_cost'] = np.where(pred['mitigated'], 0.5 * pred['actual_cost'], pred['actual_cost'])
    val_cost_by_strategy = (
        pred[pred['split'] == 'val']
        .groupby('strategy', as_index=False)['strategy_cost']
        .sum()
        .sort_values('strategy_cost', ascending=True)
    )

if len(val_cost_by_strategy) < 2:
    raise ValueError('Need at least two strategies for best-vs-baseline comparison, but only one is available after fallback.')

best_strategy = val_cost_by_strategy.iloc[0]['strategy']
baseline_strategy = val_cost_by_strategy.iloc[1]['strategy']

print('Selected best strategy:', best_strategy)
print('Selected baseline strategy:', baseline_strategy)

selected = pred[pred['strategy'].isin([best_strategy, baseline_strategy])].copy()

# Add do_nothing rows
dn = selected[['split', 'resident_id', 'week_start', 'y_claim_true', 'actual_cost']].drop_duplicates(['split', 'resident_id', 'week_start']).copy()
dn['strategy'] = 'do_nothing'
dn['y_claim_pred'] = 0
dn['mitigated'] = False
dn['strategy_cost'] = dn['actual_cost']

eval_df = pd.concat([selected, dn], ignore_index=True, sort=False)

summary = (
    eval_df
    .groupby(['split', 'strategy'], as_index=False)
    .agg(
        rows=('resident_id', 'size'),
        claim_rows=('y_claim_true', 'sum'),
        predicted_claim_rows=('y_claim_pred', 'sum'),
        mitigated_claim_rows=('mitigated', 'sum'),
        baseline_do_nothing_cost=('actual_cost', 'sum'),
        strategy_cost=('strategy_cost', 'sum'),
    )
)

summary['missed_claim_rows'] = summary['claim_rows'] - summary['mitigated_claim_rows']
summary['savings_vs_do_nothing'] = summary['baseline_do_nothing_cost'] - summary['strategy_cost']
summary['savings_pct_vs_do_nothing'] = np.where(
    summary['baseline_do_nothing_cost'] > 0,
    summary['savings_vs_do_nothing'] / summary['baseline_do_nothing_cost'],
    np.nan
)

# Keep only 3 approaches in order
order = [best_strategy, baseline_strategy, 'do_nothing']
summary['strategy'] = pd.Categorical(summary['strategy'], categories=order, ordered=True)
summary = summary.sort_values(['split', 'strategy']).reset_index(drop=True)
summary



ValueError: Need at least two strategies for best-vs-baseline comparison. Re-run ML export to generate *_business_predictions.parquet with baseline strategies.

In [ ]:
# Visual comparison
for split_name in ['val', 'test']:
    d = summary[summary['split'] == split_name].copy()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].bar(d['strategy'].astype(str), d['strategy_cost'])
    axes[0].set_title(f'{split_name.upper()} - Total Cost (Lower Better)')
    axes[0].set_xlabel('Approach')
    axes[0].set_ylabel('Cost')

    axes[1].bar(d['strategy'].astype(str), d['savings_vs_do_nothing'])
    axes[1].set_title(f'{split_name.upper()} - Savings vs Do Nothing')
    axes[1].set_xlabel('Approach')
    axes[1].set_ylabel('Savings')

    plt.tight_layout()
    plt.show()


In [ ]:
out_path = BASE_DIR / 'business_validation_summary_3_approaches.parquet'
summary.to_parquet(out_path, index=False)
print('saved:', out_path)
summary
